# Adding emotion scores

Run after `text-classification.ipynb`. Score the sentences in each description and keep the highest score for each emotion. This is a simple ranking feature, not a measured description of how every reader will feel. The model downloads on first use.


In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline

from book_recommender.analysis import calculate_max_emotion_scores

books = pd.read_csv("books_with_categories.csv", dtype={"isbn13": str})
books[["title", "description"]].head()

## Load the emotion model

CPU works on Windows and other platforms. `top_k=None` returns all emotion labels. Long inputs are truncated to the model's supported length.


In [ ]:
classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=-1,
    truncation=True,
)

## Try a short example

Match scores by their label names, since the model can return labels in different orders.


In [ ]:
example_predictions = classifier(["I love this!", "The ending was a surprise."])
calculate_max_emotion_scores(example_predictions)

## Score the book descriptions

Skip empty sentence fragments. Each book is processed once in this loop.


In [ ]:
emotion_rows = []
for _, book in tqdm(books.iterrows(), total=len(books)):
    sentences = []
    for sentence in book["description"].split("."):
        sentence = sentence.strip()
        if sentence:
            sentences.append(sentence)

    if not sentences:
        raise ValueError(f"Book {book['isbn13']} has no description sentences")

    predictions = classifier(sentences)
    scores = calculate_max_emotion_scores(predictions)
    scores["isbn13"] = book["isbn13"]
    emotion_rows.append(scores)

emotions = pd.DataFrame(emotion_rows)
emotions.head()

## Join the scores and save

The original notebook mixed up some emotion labels when sorting model outputs. Regenerate these scores before using the updated app; old scores are not a valid comparison for this version.


In [ ]:
books_with_emotions = books.merge(emotions, on="isbn13", validate="one_to_one")
books_with_emotions.to_csv("books_with_emotions.csv", index=False)
print(f"Saved {len(books_with_emotions)} books to books_with_emotions.csv")